# مرحلهٔ دوم: بهینه‌سازی پایگاه‌داده Olist

## هدف و قرارداد اجرا

هدف، بررسی صحت مسیر اجرای پنج کوئری داشبورد و اندازه‌گیری تکرارپذیر
زمان آن‌ها در PostgreSQL است. بودجهٔ هر اجرای اندازه‌گیری‌شده ۲۰۰ میلی‌ثانیه است.

این نوتبوک داده یا زمان اجرا را جعل نمی‌کند. جدول‌ها از پایگاه‌داده و
فایل‌های شواهد خوانده می‌شوند و بنچمارک جدید با پکیج مرحلهٔ اول اجرا می‌شود.

### پیش‌نیازها

- پکیج مرحلهٔ اول در محیط kernel نصب شده باشد.
- PostgreSQL و داده‌های Olist آماده باشند.
- بازسازی اولیه با فرمان زیر، از پوشهٔ مرحلهٔ اول، انجام شده باشد:

```bash
python -m data_factory load-database --config config/default.toml --reset
```

این فرمان مخرب است و اسکیمای Olist را بازسازی می‌کند؛ بنابراین نوتبوک
آن را به‌صورت خودکار اجرا نمی‌کند. اجرای دوبارهٔ نوتبوک نباید داده‌ها را حذف کند.

فایل‌های SQL و منطق بنچمارک در پروژه باقی می‌مانند؛ نوتبوک وظیفهٔ روایت،
اجرای آزمایش و نمایش شواهد را دارد.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
import json
import os
import random
import time

import pandas as pd
import sqlalchemy as sa
from IPython.display import display, Markdown

from data_factory import db_benchmark
from data_factory.config import load_config
from data_factory.database import build_database_url, OLIST_TABLES

SEED = 42
BUDGET_MS = 200
REPEATS = 15
WARMUPS = 3

random.seed(SEED)
notebook_started = time.perf_counter()

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / "stage-2-kiarash-akbari",
    cwd.parent / "stage-2-kiarash-akbari",
]

STAGE2 = next(
    (
        path for path in candidates
        if path.name == "stage-2-kiarash-akbari"
        and (path / "sql" / "queries").is_dir()
    ),
    None,
)
if STAGE2 is None:
    raise RuntimeError(
        "Open the notebook from stage 2, stage 1, or their parent directory."
    )

STAGE1 = STAGE2.parent / "stage-1-kiarash-akbari"
CONFIG_PATH = STAGE1 / "config" / "default.toml"
QUERY_DIR = STAGE2 / "sql" / "queries"
CURATED_DIR = STAGE2 / "data" / "curated"
CURATED_DIR.mkdir(parents=True, exist_ok=True)

config = load_config(CONFIG_PATH)
if config.database is None:
    raise RuntimeError("The database configuration is missing.")

password_env = config.database.password_env
if not os.environ.get(password_env):
    password = getpass(f"PostgreSQL password ({password_env}): ")
    if not password:
        raise RuntimeError("A database password is required.")
    os.environ[password_env] = password
    del password

engine = sa.create_engine(
    build_database_url(config),
    connect_args={
        "options": "-c jit=off -c work_mem=32MB -c statement_timeout=120000"
    },
)

with engine.connect() as connection:
    environment = connection.execute(sa.text(
        "SELECT current_database() AS database, "
        "current_user AS role, "
        "current_setting('jit') AS jit, "
        "current_setting('work_mem') AS work_mem, "
        "version() AS postgres_version"
    )).mappings().one()

display(pd.DataFrame([dict(environment)]))

,database,role,jit,work_mem,postgres_version
0,olist_db,kiarash,off,32MB,PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04....


## ۱. بررسی ساختار و داده‌های بارگذاری‌شده

در این بخش وجود هر نه جدول، تعداد ردیف‌های واقعی، viewها و
materialized viewها بررسی می‌شود. شمارش ردیف‌ها از پایگاه‌داده خوانده می‌شود،
نه از اعداد ثابت داخل گزارش.

همچنین فهرست SQLهای نسخه‌دار نمایش داده می‌شود تا مسیر بازسازی قابل بررسی باشد.

In [2]:
with engine.connect() as connection:
    inspector = sa.inspect(connection)
    existing_tables = set(inspector.get_table_names(schema="olist"))
    missing_tables = set(OLIST_TABLES) - existing_tables
    if missing_tables:
        raise RuntimeError(
            f"Missing Olist tables: {sorted(missing_tables)}. "
            "Run the documented database rebuild first."
        )

    table_rows = []
    for table_name in OLIST_TABLES:
        table = sa.Table(
            table_name,
            sa.MetaData(),
            schema="olist",
            autoload_with=connection,
        )
        row_count = connection.scalar(
            sa.select(sa.func.count()).select_from(table)
        )
        table_rows.append({"table": table_name, "rows": row_count})

    views = pd.read_sql_query(sa.text(
        "SELECT schemaname, viewname "
        "FROM pg_views WHERE schemaname = :schema "
        "ORDER BY viewname"
    ), connection, params={"schema": "olist"})

    snapshots = pd.read_sql_query(sa.text(
        "SELECT schemaname, matviewname, ispopulated "
        "FROM pg_matviews WHERE schemaname = :schema "
        "ORDER BY matviewname"
    ), connection, params={"schema": "olist"})

display(pd.DataFrame(table_rows))
display(views)
display(snapshots)

if snapshots.empty or not snapshots["ispopulated"].all():
    raise RuntimeError("Materialized views are missing or not populated.")

migration_files = sorted(
    path.name for path in (STAGE2 / "sql").glob("*.sql")
)
display(pd.DataFrame({"versioned_sql": migration_files}))

,table,rows
0,product_category_name_translation,71
1,customers,99441
2,sellers,3095
3,products,32951
4,geolocation,738332
5,orders,99441
6,order_items,112650
7,payments,103886
8,order_reviews,99224


,schemaname,viewname
0,olist,v_delivered_orders
1,olist,v_order_item_sales


,schemaname,matviewname,ispopulated
0,olist,mv_q1_monthly_category_sales,True
1,olist,mv_q3_delay_satisfaction,True
2,olist,mv_q4_seller_dashboard,True
3,olist,mv_q5_payment_distribution,True


,versioned_sql
0,001_create_olist_schema.sql
1,002_analytical_views.sql
2,003_functions_and_snapshots.sql
3,005_q4_seller_snapshot.sql
4,006_q5_payment_snapshot.sql
5,007_q1_monthly_category_snapshot.sql
6,008_q3_delay_satisfaction_snapshot.sql


## ۲. شواهد تاریخی و قابلیت ردیابی گزارش

نتایج آزمایش‌های قبلی از فایل‌های JSON موجود در `data/curated/` خوانده می‌شوند.
فایل‌ها به‌عنوان شواهد تاریخی نمایش داده می‌شوند، نه جایگزین اجرای زندهٔ کوئری‌ها.

نبودن یک اندازه‌گیری به معنی موفق بودن آن نیست. برای ادعاهایی مانند صحت
خروجی، هزینهٔ درج و مقایسهٔ قبل و بعد، باید فایل شاهد مناسب وجود داشته باشد.

In [3]:
artifact_paths = sorted(CURATED_DIR.glob("*.json"))
display(pd.DataFrame([
    {"artifact": path.name, "bytes": path.stat().st_size}
    for path in artifact_paths
]))

historical_rows = []
for path in artifact_paths:
    try:
        document = json.loads(path.read_text(encoding="utf-8"))
    except (ValueError, OSError):
        continue
    if not isinstance(document, dict):
        continue
    query_results = document.get("queries")
    if not isinstance(query_results, dict):
        continue
    for query_name, result in query_results.items():
        if not isinstance(result, dict) or "median_ms" not in result:
            continue
        historical_rows.append({
            "artifact": path.name,
            "query": query_name,
            "median_ms": result["median_ms"],
            "max_ms": result.get("max_ms"),
            "under_budget": result.get("all_measured_runs_under_budget"),
        })

historical_summary = pd.DataFrame(historical_rows)
display(historical_summary)

refresh_artifacts = [
    path for path in artifact_paths
    if path.name.startswith("refresh-q3-snapshot-")
]
for path in refresh_artifacts:
    display(Markdown(f"### شاهد refresh: `{path.name}`"))
    display(json.loads(path.read_text(encoding="utf-8")))

,artifact,bytes
0,benchmark-all-queries-20261028-215719.json,86191
1,benchmark-all-queries-20261028-215744.json,86187
2,benchmark-all-queries-final-20261028-221000.json,33818
3,benchmark-all-queries-final-20261028-221829.json,25232
4,benchmark-all-queries-final-20261028-224241.json,25240
5,benchmark-baseline.json,97677
6,benchmark-both-20261028-211920.json,86062
7,benchmark-final-20261028-201509-296380.json,25492
8,benchmark-final-20261028-202305-063575.json,25485
9,benchmark-jit-off-20261028-211920.json,97215


,artifact,query,median_ms,max_ms,under_budget
0,benchmark-all-queries-20261028-215719.json,q1_monthly_category_sales.sql,352.173365,376.994010,False
1,benchmark-all-queries-20261028-215719.json,q2_delivery_by_state_month.sql,138.589303,142.434967,True
2,benchmark-all-queries-20261028-215719.json,q3_delay_and_satisfaction.sql,192.754554,193.711423,True
3,benchmark-all-queries-20261028-215719.json,q4_top_sellers.sql,372.626808,376.169246,False
4,benchmark-all-queries-20261028-215719.json,q5_payment_distribution.sql,510.825954,517.043336,False
...,...,...,...,...,...
70,benchmark-top20-20261028-212458.json,q1_monthly_category_sales.sql,355.817420,366.668951,False
71,benchmark-top20-20261028-212458.json,q2_delivery_by_state_month.sql,143.010961,173.814605,True
72,benchmark-top20-20261028-212458.json,q3_delay_and_satisfaction.sql,205.566459,285.058641,False
73,benchmark-top20-20261028-212458.json,q4_top_sellers.sql,259.107101,347.806706,False


### شاهد refresh: `refresh-q3-snapshot-20261028-221811.json`

{'experiment': 'Q3 snapshot refresh cost',
 'command': 'REFRESH MATERIALIZED VIEW olist.mv_q3_delay_satisfaction',
 'concurrent': False,
 'elapsed_ms': 332.906654000908,
 'measured_at': '2026-10-28T18:48:11.081446+00:00'}

## اصلاح Q2 و وضعیت اجرای این نسخه

کوئری Q2 در فایل SQL همراه با CTE از نوع MATERIALIZED بازنویسی شده است. در آزمایش مقایسه‌ای کاربر، میانهٔ نسخهٔ اصلی 176.747 و نسخهٔ جدید 128.796 میلی‌ثانیه و بیشینهٔ نسخهٔ جدید 164.120 میلی‌ثانیه بود. خروجی‌های آزمایش برابر بودند.

این اعداد مربوط به آزمایش مقایسه‌ای‌اند، نه اجرای نهایی این نسخه. خروجی‌های قبلی برای جلوگیری از انتساب به SQL جدید پاک شده‌اند. این نسخه باید پس از جایگزینی SQL با Restart Kernel و Run All اجرا و با خروجی ذخیره شود.


## ۳. اجرای زنده و بنچمارک نهایی

هر کوئری سه warmup و پانزده اجرای اندازه‌گیری‌شده دارد.
این آزمایش warm-cache است و نباید به‌عنوان تضمین زمان در حالت cold-cache
یا تحت بار هم‌زمان تعبیر شود.

تنظیمات اتصال شامل `jit=off` و `work_mem=32MB` است.
زمان سمت کلاینت با زمان اجرای سرور یکسان نیست؛ هر دو باید جدا بررسی شوند.

In [4]:
expected_queries = {
    "q1_monthly_category_sales.sql",
    "q2_delivery_by_state_month.sql",
    "q3_delay_and_satisfaction.sql",
    "q4_top_sellers.sql",
    "q5_payment_distribution.sql",
}
actual_queries = {path.name for path in QUERY_DIR.glob("*.sql")}
if actual_queries != expected_queries:
    raise RuntimeError(
        f"Unexpected query inventory: {sorted(actual_queries)}"
    )

print("Starting benchmark...", flush=True)
benchmark_report = db_benchmark.benchmark(
    engine, QUERY_DIR, repeats=REPEATS, warmups=WARMUPS
)

benchmark_summary = pd.DataFrame([
    {
        "query": filename,
        "median_ms": result["median_ms"],
        "max_ms": result["max_ms"],
        "all_runs_under_budget": result["all_measured_runs_under_budget"],
    }
    for filename, result in sorted(benchmark_report["queries"].items())
])

display(benchmark_summary.round(3))
all_queries_pass = bool(
    benchmark_summary["all_runs_under_budget"].all()
)
print(f"All measured runs below {BUDGET_MS} ms: {all_queries_pass}")

Starting benchmark...


,query,median_ms,max_ms,all_runs_under_budget
0,q1_monthly_category_sales.sql,11.994,16.128,True
1,q2_delivery_by_state_month.sql,127.679,181.523,True
2,q3_delay_and_satisfaction.sql,1.215,2.245,True
3,q4_top_sellers.sql,4.523,5.941,True
4,q5_payment_distribution.sql,3.525,4.089,True


All measured runs below 200 ms: True


## ۴. برنامه‌های اجرای نهایی

برنامه‌های JSON تولیدشده توسط بنچمارک نمایش داده می‌شوند تا مسیر scan،
join و aggregation قابل بررسی باشد. وجود یک پلن نهایی به‌تنهایی ثابت نمی‌کند
که بهینه‌سازی مؤثر بوده است؛ برای هر کوئری باید آن را با پلن اولیهٔ ذخیره‌شده
مقایسه کرد.

در این بخش فقط دادهٔ موجود نمایش داده می‌شود و کلیدهای ناموجود جعل نمی‌شوند.

In [5]:
for query_name, result in sorted(benchmark_report["queries"].items()):
    display(Markdown(f"### `{query_name}`"))
    plan_fields = {
        key: value
        for key, value in result.items()
        if "plan" in key.lower() or "explain" in key.lower()
    }
    if plan_fields:
        display(plan_fields)
    else:
        print("No plan field found. Available result keys:", sorted(result))

### `q1_monthly_category_sales.sql`

{'explain_analyze': [{'Plan': {'Node Type': 'Sort',
    'Parallel Aware': False,
    'Async Capable': False,
    'Startup Cost': 92.0,
    'Total Cost': 95.2,
    'Plan Rows': 1282,
    'Plan Width': 46,
    'Actual Startup Time': 0.697,
    'Actual Total Time': 0.8,
    'Actual Rows': 1282,
    'Actual Loops': 1,
    'Sort Key': ['purchase_month', 'product_category_name'],
    'Sort Method': 'quicksort',
    'Sort Space Used': 146,
    'Sort Space Type': 'Memory',
    'Shared Hit Blocks': 13,
    'Shared Read Blocks': 0,
    'Shared Dirtied Blocks': 0,
    'Shared Written Blocks': 0,
    'Local Hit Blocks': 0,
    'Local Read Blocks': 0,
    'Local Dirtied Blocks': 0,
    'Local Written Blocks': 0,
    'Temp Read Blocks': 0,
    'Temp Written Blocks': 0,
    'Plans': [{'Node Type': 'Seq Scan',
      'Parent Relationship': 'Outer',
      'Parallel Aware': False,
      'Async Capable': False,
      'Relation Name': 'mv_q1_monthly_category_sales',
      'Alias': 'mv_q1_monthly_category_s

### `q2_delivery_by_state_month.sql`

{'explain_analyze': [{'Plan': {'Node Type': 'Sort',
    'Parallel Aware': False,
    'Async Capable': False,
    'Startup Cost': 21976.98,
    'Total Cost': 22211.11,
    'Plan Rows': 93654,
    'Plan Width': 64,
    'Actual Startup Time': 147.738,
    'Actual Total Time': 147.762,
    'Actual Rows': 556,
    'Actual Loops': 1,
    'Sort Key': ['grouped.purchase_month', 'grouped.customer_state'],
    'Sort Method': 'quicksort',
    'Sort Space Used': 57,
    'Sort Space Type': 'Memory',
    'Shared Hit Blocks': 3259,
    'Shared Read Blocks': 0,
    'Shared Dirtied Blocks': 0,
    'Shared Written Blocks': 0,
    'Local Hit Blocks': 0,
    'Local Read Blocks': 0,
    'Local Dirtied Blocks': 0,
    'Local Written Blocks': 0,
    'Temp Read Blocks': 0,
    'Temp Written Blocks': 0,
    'Plans': [{'Node Type': 'Aggregate',
      'Strategy': 'Hashed',
      'Partial Mode': 'Simple',
      'Parent Relationship': 'InitPlan',
      'Subplan Name': 'CTE grouped',
      'Parallel Aware': False,


### `q3_delay_and_satisfaction.sql`

{'explain_analyze': [{'Plan': {'Node Type': 'Sort',
    'Parallel Aware': False,
    'Async Capable': False,
    'Startup Cost': 1.08,
    'Total Cost': 1.09,
    'Plan Rows': 4,
    'Plan Width': 108,
    'Actual Startup Time': 0.027,
    'Actual Total Time': 0.028,
    'Actual Rows': 4,
    'Actual Loops': 1,
    'Sort Key': ['bucket'],
    'Sort Method': 'quicksort',
    'Sort Space Used': 25,
    'Sort Space Type': 'Memory',
    'Shared Hit Blocks': 1,
    'Shared Read Blocks': 0,
    'Shared Dirtied Blocks': 0,
    'Shared Written Blocks': 0,
    'Local Hit Blocks': 0,
    'Local Read Blocks': 0,
    'Local Dirtied Blocks': 0,
    'Local Written Blocks': 0,
    'Temp Read Blocks': 0,
    'Temp Written Blocks': 0,
    'Plans': [{'Node Type': 'Seq Scan',
      'Parent Relationship': 'Outer',
      'Parallel Aware': False,
      'Async Capable': False,
      'Relation Name': 'mv_q3_delay_satisfaction',
      'Alias': 'mv_q3_delay_satisfaction',
      'Startup Cost': 0.0,
      'Total

### `q4_top_sellers.sql`

{'explain_analyze': [{'Plan': {'Node Type': 'Limit',
    'Parallel Aware': False,
    'Async Capable': False,
    'Startup Cost': 157.77,
    'Total Cost': 157.82,
    'Plan Rows': 20,
    'Plan Width': 87,
    'Actual Startup Time': 1.537,
    'Actual Total Time': 1.543,
    'Actual Rows': 20,
    'Actual Loops': 1,
    'Shared Hit Blocks': 46,
    'Shared Read Blocks': 0,
    'Shared Dirtied Blocks': 0,
    'Shared Written Blocks': 0,
    'Local Hit Blocks': 0,
    'Local Read Blocks': 0,
    'Local Dirtied Blocks': 0,
    'Local Written Blocks': 0,
    'Temp Read Blocks': 0,
    'Temp Written Blocks': 0,
    'Plans': [{'Node Type': 'Sort',
      'Parent Relationship': 'Outer',
      'Parallel Aware': False,
      'Async Capable': False,
      'Startup Cost': 157.77,
      'Total Cost': 165.4,
      'Plan Rows': 3053,
      'Plan Width': 87,
      'Actual Startup Time': 1.536,
      'Actual Total Time': 1.538,
      'Actual Rows': 20,
      'Actual Loops': 1,
      'Sort Key': ['reve

### `q5_payment_distribution.sql`

{'explain_analyze': [{'Plan': {'Node Type': 'Sort',
    'Parallel Aware': False,
    'Async Capable': False,
    'Startup Cost': 1.91,
    'Total Cost': 1.98,
    'Plan Rows': 27,
    'Plan Width': 196,
    'Actual Startup Time': 0.033,
    'Actual Total Time': 0.036,
    'Actual Rows': 27,
    'Actual Loops': 1,
    'Sort Key': ['payment_type', 'payment_installments'],
    'Sort Method': 'quicksort',
    'Sort Space Used': 28,
    'Sort Space Type': 'Memory',
    'Shared Hit Blocks': 1,
    'Shared Read Blocks': 0,
    'Shared Dirtied Blocks': 0,
    'Shared Written Blocks': 0,
    'Local Hit Blocks': 0,
    'Local Read Blocks': 0,
    'Local Dirtied Blocks': 0,
    'Local Written Blocks': 0,
    'Temp Read Blocks': 0,
    'Temp Written Blocks': 0,
    'Plans': [{'Node Type': 'Seq Scan',
      'Parent Relationship': 'Outer',
      'Parallel Aware': False,
      'Async Capable': False,
      'Relation Name': 'mv_q5_payment_distribution',
      'Alias': 'mv_q5_payment_distribution',
   

## ۵. فضای ایندکس‌ها

ایندکس هزینهٔ ذخیره‌سازی و نگهداری دارد. این جدول اندازهٔ واقعی ایندکس‌ها
و نسبت اندازهٔ آن‌ها به جدول را نشان می‌دهد.

این اندازه‌گیری جایگزین آزمون قبل/بعد سرعت درج نیست؛ هزینهٔ نوشتن باید در
یک آزمایش مستقل و کنترل‌شده اندازه‌گیری و به فایل شاهد مرتبط ارجاع داده شود.

In [6]:
with engine.connect() as connection:
    index_sizes = pd.read_sql_query(sa.text(
        "SELECT t.relname AS table_name, i.relname AS index_name, "
        "pg_relation_size(i.oid) AS index_bytes, "
        "pg_relation_size(t.oid) AS table_bytes, "
        "ROUND(pg_relation_size(i.oid)::numeric / "
        "NULLIF(pg_relation_size(t.oid), 0), 4) AS index_to_table_ratio "
        "FROM pg_index x "
        "JOIN pg_class t ON t.oid = x.indrelid "
        "JOIN pg_class i ON i.oid = x.indexrelid "
        "JOIN pg_namespace n ON n.oid = t.relnamespace "
        "WHERE n.nspname = :schema "
        "ORDER BY t.relname, i.relname"
    ), connection, params={"schema": "olist"})
display(index_sizes)

,table_name,index_name,index_bytes,table_bytes,index_to_table_ratio
0,customers,pk_customers,7856128,11878400,0.6614
1,geolocation,pk_geolocation,16605184,58286080,0.2849
2,mv_q1_monthly_category_sales,ux_mv_q1_monthly_category_sales,73728,106496,0.6923
3,mv_q3_delay_satisfaction,ux_mv_q3_delay_satisfaction_bucket,16384,8192,2.0000
4,mv_q4_seller_dashboard,ux_mv_q4_seller_dashboard_seller,196608,376832,0.5217
5,mv_q5_payment_distribution,ux_mv_q5_payment_distribution,16384,8192,2.0000
6,order_items,pk_order_items,6668288,18833408,0.3541
7,order_reviews,pk_order_reviews,12197888,14721024,0.8286
8,orders,pk_orders,7864320,14819328,0.5307
9,payments,pk_payments,8093696,9674752,0.8366


## ۶. محدودیت‌ها و دامنهٔ اعتبار

- عبور از بودجه در این اجرا، تضمین عملکرد روی هر سخت‌افزار یا تحت هر بار نیست.
- دادهٔ materialized view تا refresh بعدی ممکن است قدیمی باشد.
- seed ثابت، زمان اجرای پایگاه‌داده را قطعی نمی‌کند؛ نتایج معنایی باید ثابت
  بمانند اما زمان‌ها با بار سیستم و وضعیت cache تغییر می‌کنند.
- بنچمارک موفق به‌تنهایی صحت معنایی کوئری را ثابت نمی‌کند.
- این نوتبوک هنوز جایگزین شواهد لازم برای Procedure، Function،
  idempotency، هزینهٔ درج و مقایسهٔ view با snapshot نیست.
  هر بخش باید با آزمون یا artifact واقعی پروژه تکمیل شود.

## ۷. ذخیرهٔ نتیجه و جمع‌بندی

فایل خروجی پیش از کنترل نهایی بودجه ذخیره می‌شود؛ بنابراین اگر کوئری‌ای
از بودجه عبور کند، شاهد شکست نیز باقی می‌ماند.

اعداد گزارش باید به فایل مشخص و خروجی اجراشدهٔ همین نوتبوک ارجاع دهند؛
نباید اعداد قدیمی با اجرای جدید بدون توضیح جایگزین شوند.

In [7]:
benchmark_report["experiment"] = {
    "name": "Stage 2 notebook live benchmark",
    "query_directory": "sql/queries",
    "repeats": REPEATS,
    "warmups": WARMUPS,
    "budget_ms": BUDGET_MS,
    "cache_protocol": "warmups; cache not flushed",
    "connection_settings": {
        "jit": "off",
        "work_mem": "32MB",
        "statement_timeout": "120s",
    },
    "seed": SEED,
    "measured_at": datetime.now(timezone.utc).isoformat(),
}

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output_path = CURATED_DIR / f"benchmark-final-{timestamp}.json"
with output_path.open("x", encoding="utf-8") as handle:
    json.dump(benchmark_report, handle, ensure_ascii=False, indent=2)
    handle.write("\n")

print("Saved:", output_path.relative_to(STAGE2))
print("Notebook elapsed seconds:", round(time.perf_counter() - notebook_started, 2))
engine.dispose()

assert all_queries_pass, (
    "At least one measured run exceeded the budget; "
    "inspect the saved artifact instead of discarding this run."
)

Saved: data/curated/benchmark-final-20261028-203102-777061.json
Notebook elapsed seconds: 8.49


## AI usage

از دستیار هوش مصنوعی برای تهیه و اصلاح سازندهٔ نوتبوک، تشخیص خطای مسیرها،
هماهنگ‌کردن کلیدهای خروجی بنچمارک و تنظیم روایت کمک گرفته شد.
بخش‌هایی از کد نیز توسط دستیار پیشنهاد شد.

اجرای واقعی، بازبینی خروجی‌ها و تأیید ادعاهای گزارش باید توسط صاحب پروژه
انجام شود. تولید فایل نوتبوک به معنی اجرا یا تأیید کامل الزامات مرحله نیست.

در این اصلاح، دستیار نسخهٔ آزمایش‌شدهٔ Q2 را استخراج کرد، سلول‌های موقت عیب‌یابی را جدا کرد و فایل آمادهٔ اجرای مجدد ساخت. اجرای نهایی این نسخه توسط دستیار انجام نشده است.
